# Speak to the notebook

[Back to the introduction](ch01_demo.ipynb). Run in order. This notebook records your voice with the device's microphone and transcribes it in the browser, so nothing leaves the page. It works on a laptop or a tablet: the page must be served over HTTPS (or localhost), and the browser asks for microphone permission the first time you tap Record.

A notebook kernel runs in a Web Worker, which cannot open the microphone. The `AudioRecorder` widget therefore records on the page itself and hands the kernel a 16 kHz mono WAV file, which a speech recognition pipeline takes as bytes. Outside JupyterLite, install it with `pip install 'lmlite[audio]'`.

In [ ]:
from lmlite.audio import AudioRecorder

recorder = AudioRecorder()
recorder

Tap **Record**, say a sentence in English, then tap **Stop**. The next cell waits for the recording.

If the widget says the recording is silent, or you hear nothing, tick **Raw input** and record again. By default the browser processes the microphone (echo cancellation, noise suppression, gain control), and on some devices that processing mixes the input down to silence; Raw input turns it off and levels the recording instead. Also check that the browser may use the microphone and that the right input is selected in the system settings.

In [ ]:
from IPython.display import Audio

wav = await recorder.wait()
info = recorder.info
print(f"{info['seconds']:.1f} s of audio, {len(wav)} bytes, peak {info['peak']:.2f} "
      f"({info['channels']} channel(s), {info['sampleRate']} Hz, raw input: {info['raw']})")
if info["peak"] < 0.005:
    print("This recording is silent: tick Raw input, run recorder.clear(), record again and rerun this cell.")
Audio(wav)

## Transcribe it

`whisper-tiny.en` is an English-only model of about 40 MB. Recordings longer than 30 seconds need `chunk_length_s=30`. Tap Record again and rerun the cells from `wav = await recorder.wait()` after `recorder.clear()` to try another sentence.

In [ ]:
from lmlite import LLM

transcriber = await LLM.pipeline("automatic-speech-recognition", model="whisper-tiny.en", backend="wasm")
result = await transcriber(wav)
print(result["text"])
await transcriber.dispose()